In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import XLNetTokenizer
import sys
import os

# add src to path
sys.path.append(os.path.abspath('..'))

from src.dataset import get_datasets
from src.model import DualHeadXLNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

c:\Users\imada\Uni\Master\1. Semester\Introduction to Natural Language Processing\NLPGroup29\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cpu


In [ ]:
# init tokenizer
tokenizer = XLNetTokenizer.from_pretrained('xlnet-base-cased')

# load real data
train_ds, test_ds = get_datasets(
    '../data/processed/train.csv', 
    '../data/processed/test.csv', 
    tokenizer
)

# pick subset from TRAIN DS
mini_batch = torch.utils.data.Subset(train_ds, range(8))
loader = DataLoader(mini_batch, batch_size=4, shuffle=True)

# inspect one batch
batch = next(iter(loader))
print("input shape:", batch['input_ids'].shape)
print("clarity labels:", batch['clarity_labels'])
print("evasion labels:", batch['evasion_labels'])

c:\Users\imada\Uni\Master\1. Semester\Introduction to Natural Language Processing\NLPGroup29\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\imada\.cache\huggingface\hub\models--xlnet-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


applying voting logic...
splitting data (total: 3448)...
input shape: torch.Size([4, 512])
clarity labels: tensor([1, 2, 0, 0])
evasion labels: tensor([1, 7, 0, 0])


In [3]:
# init model
model = DualHeadXLNet().to(device)
model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4) # high lr for fast overfit
loss_fn = nn.CrossEntropyLoss()

print("starting sanity check...")

for epoch in range(30):
    total_loss = 0
    for batch in loader:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        c_true = batch['clarity_labels'].to(device)
        e_true = batch['evasion_labels'].to(device)
        
        optimizer.zero_grad()
        
        c_logits, e_logits = model(ids, mask)
        
        loss = loss_fn(c_logits, c_true) + loss_fn(e_logits, e_true)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    if epoch % 5 == 0:
        print(f"epoch {epoch}: loss {total_loss:.4f}")

print(f"final loss: {total_loss:.4f}")
# should be < 0.1

loading xlnet-base-cased...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


starting sanity check...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


epoch 0: loss 8.5055
epoch 5: loss 0.6680
epoch 10: loss 0.0028


KeyboardInterrupt: 